<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/PracticaMovies_Clase26Ago.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Práctica Movies**

* Asignatura: Sistemas de recomendación
* Profesor: Dra. Karla Olmos Sánchez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* Práctica en clase 26 de agosto de 2026


Equipo I
Integrantes
Eduardo Sebastian Delgado Andrade
Álvaro Hernández Jarquín
Ingrid Margarita Terrazas Alvarado


## Propósito de la práctica
El objetivo no es memorizar instrucciones de Python, sino comprender la lógica de construcción de un perfil de usuario. Cada equipo recibirá todos los fragmentos de código necesarios, pero estarán deliberadamente desordenados. Deberán identificar la función de cada fragmento, establecer dependencias entre pasos, proponer el orden correcto, ejecutar la solución y justificar qué información aporta cada resultado.
## Al finalizar, el equipo deberá ser capaz de:
* Distinguir datos del usuario, datos del ítem y valoraciones.
*	Explicar por qué es necesario integrar ratings y características de las películas.
*	Transformar los géneros de una película para analizarlos individualmente.
*	Construir un perfil de preferencias combinando frecuencia y valoración promedio.
*	Interpretar el perfil resultante sin confundir “género más visto” con “género más preferido”.
## Dinámica de trabajo
1. **Analicen antes de ejecutar**. No comiencen copiando los fragmentos a Colab. Primero lean todos y discutan qué hace cada uno.
2. **Construyan el orden**. En la tabla de secuencia anoten las letras de los fragmentos en el orden que consideren correcto.
3. **Justifiquen dependencias**. Para cada paso expliquen qué información necesita recibir del paso anterior y qué produce.
4. **Ejecuten y verifiquen**. Una vez que el equipo acuerde el orden, ejecuten el código. Si aparece un error, revisen la lógica antes de cambiar instrucciones.
5. **Interpreten**. El resultado final debe poder explicarse conceptualmente: ¿qué sabemos ahora del usuario que no sabíamos al inicio?
## Situación
Se utilizarán los archivos ratings.csv y movies.csv de MovieLens. El equipo analizará un usuario específico y construirá un perfil de preferencias por género. Pueden trabajar inicialmente con el usuario 414 o sustituirlo por el usuario indicado por la profesora.
User => rating => movie => genre
Number interactions
## Antes del código: reconstruyan el proceso conceptual
Sin consultar todavía los fragmentos, escriban qué creen que debería ocurrir entre los datos originales y el perfil final:

| Etapa | ¿Qué debe hacerse? | ¿Por qué es necesaria? |
|-------|---------------------|-------------------------|
| 1     | Seleccionar usuario | Para crear el perfil |
| 2     | Unir ratings + peliculas | Para tener los datos de la película calificada |
| 3     | Separar por géneros | Para resolver el problema de varios géneros en un solo registro |
| 4     | Agrupar por géneros | Para tener el conteo por género |
| 5     | Normalizar frecuencia | Para tener los valores >=1 de la revisión de películas |
| 6     | Calcular preferencia | Multiplicamos la frecuencia normalizada con el promedio del rating |


In [ ]:
# Importar librerías

import pandas as pd

In [ ]:
# Cargar bases de datos

ratings = pd.read_csv("/content/drive/MyDrive/MIAAD_05_SistemasDeRecomendacion/rating.csv")
movies = pd.read_csv("/content/drive/MyDrive/MIAAD_05_SistemasDeRecomendacion/movie.csv")

In [ ]:
# Imprimir encabezados

print(ratings.head())
print(movies.head())
print(ratings.columns)
print(movies.columns)

   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')
Index(['movieId', 'title', 'genres'], dtype='obje

In [ ]:
user_id = 414

user_profile = ratings[ratings["userId"] == user_id].copy()

print(user_profile.head())
print("Número de valoraciones:", len(user_profile))

       userId  movieId  rating            timestamp
57515     414       10     2.0  1996-08-30 16:26:29
57516     414       11     4.0  1996-08-30 16:28:50
57517     414       21     4.0  1996-08-30 16:27:41
57518     414       25     5.0  1996-08-30 16:30:34
57519     414       32     4.0  1996-08-30 16:28:19
Número de valoraciones: 65


In [ ]:
user_profile = user_profile.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

print(user_profile[["movieId", "title", "rating", "genres"]].head())

   movieId                                      title  rating  \
0       10                           GoldenEye (1995)     2.0   
1       11             American President, The (1995)     4.0   
2       21                          Get Shorty (1995)     4.0   
3       25                   Leaving Las Vegas (1995)     5.0   
4       32  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)     4.0   

                      genres  
0  Action|Adventure|Thriller  
1       Comedy|Drama|Romance  
2      Comedy|Crime|Thriller  
3              Drama|Romance  
4    Mystery|Sci-Fi|Thriller  


In [ ]:
genres_profile = user_profile.copy()
genres_profile["genres"] = genres_profile["genres"].str.split("|")
genres_profile = genres_profile.explode("genres")

print(genres_profile[["movieId", "title", "rating", "genres"]].head(10))

   movieId                           title  rating     genres
0       10                GoldenEye (1995)     2.0     Action
0       10                GoldenEye (1995)     2.0  Adventure
0       10                GoldenEye (1995)     2.0   Thriller
1       11  American President, The (1995)     4.0     Comedy
1       11  American President, The (1995)     4.0      Drama
1       11  American President, The (1995)     4.0    Romance
2       21               Get Shorty (1995)     4.0     Comedy
2       21               Get Shorty (1995)     4.0      Crime
2       21               Get Shorty (1995)     4.0   Thriller
3       25        Leaving Las Vegas (1995)     5.0      Drama


In [ ]:
genre_ratings = (
    genres_profile.groupby("genres")
    .agg(
        peliculas=("movieId", "count"),
        rating_promedio=("rating", "mean")
    )
    .sort_values("rating_promedio", ascending=False)
)

print(genre_ratings)

             peliculas  rating_promedio
genres                                 
IMAX                 3         5.000000
Horror               1         5.000000
War                  4         4.250000
Musical              6         4.000000
Documentary          1         4.000000
Animation            6         4.000000
Drama               29         3.931034
Romance             18         3.888889
Crime                8         3.750000
Thriller            19         3.631579
Children            10         3.600000
Fantasy              6         3.500000
Adventure           15         3.466667
Comedy              26         3.423077
Action              16         3.375000
Mystery              3         3.333333
Sci-Fi               7         3.285714
Western              2         3.000000


In [ ]:
genre_ratings["frecuencia_normalizada"] = (
    genre_ratings["peliculas"] / genre_ratings["peliculas"].max()
)

genre_ratings["preferencia"] = (
    genre_ratings["frecuencia_normalizada"] *
    genre_ratings["rating_promedio"]
)

In [ ]:
perfil_final = genre_ratings.sort_values(
    "preferencia", ascending=False
)

print(perfil_final[["peliculas", "rating_promedio",
                    "frecuencia_normalizada", "preferencia"]])

             peliculas  rating_promedio  frecuencia_normalizada  preferencia
genres                                                                      
Drama               29         3.931034                1.000000     3.931034
Comedy              26         3.423077                0.896552     3.068966
Romance             18         3.888889                0.620690     2.413793
Thriller            19         3.631579                0.655172     2.379310
Action              16         3.375000                0.551724     1.862069
Adventure           15         3.466667                0.517241     1.793103
Children            10         3.600000                0.344828     1.241379
Crime                8         3.750000                0.275862     1.034483
Musical              6         4.000000                0.206897     0.827586
Animation            6         4.000000                0.206897     0.827586
Sci-Fi               7         3.285714                0.241379     0.793103

In [ ]:
print("Géneros con mayor preferencia del usuario:")
print(perfil_final.head(5))

Géneros con mayor preferencia del usuario:
          peliculas  rating_promedio  frecuencia_normalizada  preferencia
genres                                                                   
Drama            29         3.931034                1.000000     3.931034
Comedy           26         3.423077                0.896552     3.068966
Romance          18         3.888889                0.620690     2.413793
Thriller         19         3.631579                0.655172     2.379310
Action           16         3.375000                0.551724     1.862069


Hoja de reconstrucción
Antes de ejecutar, escriban la secuencia de letras que propone el equipo y completen la justificación.
| Paso | Fragmento | ¿Qué hace? | ¿De qué paso/variable depende? |
|------|-----------|------------|--------------------------------|
| 1    | B         | Importar librerías | De que esté instalada |
| 2    | A         | Cargar bases de datos | De que las bases de datos estén en el mismo archivo y que se importe la librería |
| 3    | H         | Exploración Datos | Paso 1 y 2 |
| 4    | G         | Selección del usuario | Paso 2 |
| 5    | C         | Unir ratings + películas | Paso 4 |
| 6    | E         | Separar por géneros | Paso 5 |
| 7    | I         | Agrupar por géneros | Paso 6 |
| 8    | D         | Normalizar frecuencia | Paso 7 |
| 9    | F         | Calcular preferencia | Paso 8 |
| 10   | J         | Ordenar e interpretar | Paso 9 |



## Preguntas de análisis durante la ejecución
1. ¿Por qué no podemos analizar las preferencias por género utilizando únicamente ratings.csv?

Necesitamos cruzar la información con las películas para obtener sus géneros

2. ¿Qué representa user_profile antes del merge y qué información adicional adquiere después del merge?

Solo obtenemos la calificacion y el Id del usuario (ratings)

3. Una película puede tener varios géneros. ¿Qué problema resolvería str.split("|") seguido de explode("genres")?

Cada película tiene varios géneros en un solo registro, se necesita separarlos

4. Después de explode(), ¿una película puede aparecer en varias filas? ¿Por qué eso es útil para este análisis?

Se puede apreciar cada género individualmente

5. ¿Qué diferencia existe entre peliculas y rating_promedio dentro de genre_ratings?

peliculas se refiere a el número de películas calificadas de cierto género y el rating_promedio es el promedio de dichas películas

6. ¿Por qué dividimos el número de películas de cada género entre el máximo número de películas de algún género?

Para normalizarlo

7. ¿Qué significa que frecuencia_normalizada sea igual a 1 para un género?

Es el género que más se calificó

8. ¿Por qué la preferencia multiplica frecuencia_normalizada por rating_promedio?

Para generar un valor que tome en cuenta su rating con la cantidad de interacciones

9. ¿Puede un género tener rating_promedio alto pero preferencia relativamente baja? Expliquen un caso.

Sí se puede ya que hay casos donde el rating es alto pero hay pocas interacciones

10. ¿Cuál sería el riesgo de recomendar únicamente usando el género con mayor rating_promedio?

Podría haber un sesgo por pocas interacciones


**Uso responsable de la IA:** Se le pidió a DeepSeek-V3.2 cambiar el formato de Markdown para poner las tablas de este documento.